In [1]:
# Load environment and suppress warnings

import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-5-mini", temperature=0)
else:
    model = None
    print("Skipping model init. No API key found.")


In [3]:
# Define a simple triage prompt (single responsibility)

triage_prompt = PromptTemplate.from_template(
"""
Classify the support ticket into exactly ONE category:

Account
Billing
Bug
Outage
Other

Return only the category name.

Ticket:
{ticket_text}
"""
)

print("Prompt template created.")

Prompt template created.


In [4]:
# Build a reusable chain component (prompt -> model -> string output)

parser = StrOutputParser()

triage_chain = triage_prompt | model | parser

print("Chain created:", triage_chain)

Chain created: first=PromptTemplate(input_variables=['ticket_text'], input_types={}, partial_variables={}, template='\nClassify the support ticket into exactly ONE category:\n\nAccount\nBilling\nBug\nOutage\nOther\n\nReturn only the category name.\n\nTicket:\n{ticket_text}\n') middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-openai': '1.8.5'}}, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool

In [5]:
# Execute the same chain with different inputs (same structure, different data)

ticket_1 = "Customer cannot reset password and reports login failure after multiple attempts."
ticket_2 = "Billing page shows incorrect subscription amount and the last payment was charged twice."

result_1 = triage_chain.invoke({"ticket_text": ticket_1})
result_2 = triage_chain.invoke({"ticket_text": ticket_2})

print("=== RESULT 1 ===")
print(result_1)
print()

print("=== RESULT 2 ===")
print(result_2)

=== RESULT 1 ===
Account

=== RESULT 2 ===
Billing
